In [1]:
from fixture.factory.dataset.ohlcv import factory_ohlcv_cycle

df = factory_ohlcv_cycle()
df

Date,Open,High,Low,Close,Volume
datetime[μs],f64,f64,f64,f64,i64
2000-01-01 00:00:00,101.0,101.28,99.5,99.51,33056
2000-01-02 00:00:00,99.51,100.17,99.3,99.98,42437
2000-01-03 00:00:00,99.98,100.26,99.34,99.69,20057
2000-01-04 00:00:00,98.69,101.11,98.53,101.01,19700
2000-01-05 00:00:00,101.01,101.27,100.66,100.82,15548
…,…,…,…,…,…
2000-04-05 00:00:00,108.22,109.21,107.96,109.12,12964
2000-04-06 00:00:00,109.12,109.23,107.7,107.94,14345
2000-04-07 00:00:00,107.94,109.78,107.74,109.6,24034


In [2]:
from feature.closes.service import derive_closes


closes = derive_closes(df, 8)
closes

Date,now,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64
2000-01-10 00:00:00,-40.217819,72.706156,113.051581,-18.930908,-35.771103,-18.827732,131.541508,-29.04795,47.120244
2000-01-11 00:00:00,35.321858,-40.217819,72.706156,113.051581,-18.930908,-35.771103,-18.827732,131.541508,-29.04795
2000-01-12 00:00:00,22.501599,35.321858,-40.217819,72.706156,113.051581,-18.930908,-35.771103,-18.827732,131.541508
2000-01-13 00:00:00,-18.584639,22.501599,35.321858,-40.217819,72.706156,113.051581,-18.930908,-35.771103,-18.827732
2000-01-14 00:00:00,-22.543504,-18.584639,22.501599,35.321858,-40.217819,72.706156,113.051581,-18.930908,-35.771103
…,…,…,…,…,…,…,…,…,…
2000-04-05 00:00:00,182.184722,16.813006,58.128797,16.939586,107.955594,-37.063478,0.0,168.360587,31.885625
2000-04-06 00:00:00,-108.726769,182.184722,16.813006,58.128797,16.939586,107.955594,-37.063478,0.0,168.360587
2000-04-07 00:00:00,152.618573,-108.726769,182.184722,16.813006,58.128797,16.939586,107.955594,-37.063478,0.0


In [3]:
from feature.closes.derive import derive_closes_n4

closes_n4 = derive_closes_n4(df)
closes_n4

Date,now,lag_1,lag_2,lag_3,lag_4
datetime[μs],f64,f64,f64,f64,f64
2000-01-06 00:00:00,-35.771103,-18.827732,131.541508,-29.04795,47.120244
2000-01-07 00:00:00,-18.930908,-35.771103,-18.827732,131.541508,-29.04795
2000-01-08 00:00:00,113.051581,-18.930908,-35.771103,-18.827732,131.541508
2000-01-09 00:00:00,72.706156,113.051581,-18.930908,-35.771103,-18.827732
2000-01-10 00:00:00,-40.217819,72.706156,113.051581,-18.930908,-35.771103
…,…,…,…,…,…
2000-04-05 00:00:00,182.184722,16.813006,58.128797,16.939586,107.955594
2000-04-06 00:00:00,-108.726769,182.184722,16.813006,58.128797,16.939586
2000-04-07 00:00:00,152.618573,-108.726769,182.184722,16.813006,58.128797


In [4]:
closes.equals(closes_n4)

False

In [5]:
# closesの特徴量分析

import plotly.express as px
import plotly.graph_objects as go
import polars as pl

# 相関行列ヒートマップ
corr = closes.drop("Date").to_pandas().corr()
fig = px.imshow(
    corr,
    labels=dict(x="Features", y="Features", color="Correlation"),
    x=corr.columns,
    y=corr.columns,
    title="特徴量間の相関関係ヒートマップ",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1
)
fig.update_layout(width=800, height=800)
fig.show()

# 特徴量分布のボックスプロット
fig = go.Figure()
for col in closes.columns[1:]:  # Dateを除外
    fig.add_trace(go.Box(
        y=closes[col],
        name=col,
        boxpoints='outliers',
        jitter=0.3,
        pointpos=-1.8
    ))
fig.update_layout(
    title="特徴量の分布と外れ値",
    yaxis_title="Value",
    boxmode='group'
)
fig.show()

# 時系列プロット（now特徴量）
fig = px.line(
    closes.to_pandas(),
    x="Date",
    y="now",
    title="'now'特徴量の時系列変化",
    labels={"now": "Value"},
    template="plotly_white"
)
# Polarsのrolling_meanを使用
rolling_mean = (
    closes.select(
        pl.col("now").rolling_mean(window_size=5).alias("now_ma5")
    )
)
fig.add_trace(go.Scatter(
    x=closes["Date"],
    y=rolling_mean["now_ma5"],
    mode="lines",
    name="5日移動平均",
    line=dict(color="red", dash="dot")
))
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Value",
    hovermode="x unified"
)
fig.show()

# ラグ特徴量の相互作用（3D散布図）
fig = px.scatter_3d(
    closes.head(100).to_pandas(),
    x='lag_1',
    y='lag_2',
    z='now',
    color='lag_3',
    title="ラグ特徴量の3D相互作用",
    labels={'lag_1': 'Lag 1', 'lag_2': 'Lag 2', 'now': 'Current'},
    color_continuous_scale=px.colors.sequential.Viridis
)
fig.update_layout(
    scene=dict(
        xaxis_title='Lag 1',
        yaxis_title='Lag 2',
        zaxis_title='Current Value'
    ),
    width=1000,
    height=800
)
fig.show()